# 4. Lipstick on a Pig: debiasing that only hides the bias

**Claim under test:** you can remove gender bias from word embeddings by
identifying the gender direction and projecting it out.

That method is Bolukbasi et al. (2016), and it works, in the narrow sense that
the metric it optimises goes to zero. Gonen & Goldberg (2019) showed the bias is
still entirely there. The debiasing removed the *measurement*, not the
information.

This is the same failure as Filter 2 in the flowchart, in a domain where it can
be shown exactly rather than argued about.

- Gonen & Goldberg, *Lipstick on a Pig* (`papers/gonen-2019-lipstick-on-a-pig.pdf`)
- Bolukbasi et al., *Man is to Computer Programmer as Woman is to Homemaker* (2016)

Runs on GloVe 300d, 400k vectors, downloaded on first use by `gensim`.

In [1]:
import numpy as np
import gensim.downloader as api
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

kv = api.load('glove-wiki-gigaword-300')

VOCAB = 40000
words = [w for w in kv.index_to_key[:VOCAB] if w.isalpha() and len(w) >= 2]   # >= 2 keeps 'he'
E = np.array([kv[w] for w in words])
E = E / np.linalg.norm(E, axis=1, keepdims=True)
index = {w: i for i, w in enumerate(words)}
print(f'{len(words)} words, {E.shape[1]} dimensions')

36849 words, 300 dimensions


## Step 1: find the gender direction

Bolukbasi's method. Take pairs that differ only in gender, take their difference
vectors, and take the first principal component. That single direction is what
the debiasing will remove.

In [2]:
PAIRS = [('he', 'she'), ('his', 'her'), ('man', 'woman'), ('men', 'women'),
         ('male', 'female'), ('boy', 'girl'), ('father', 'mother'),
         ('son', 'daughter'), ('brother', 'sister'), ('himself', 'herself')]

diffs = np.array([E[index[a]] - E[index[b]] for a, b in PAIRS
                  if a in index and b in index])
_, S, Vt = np.linalg.svd(diffs - diffs.mean(0))
g = Vt[0] / np.linalg.norm(Vt[0])
if E[index['he']] @ g < 0:          # orient so that positive = male
    g = -g

print(f'{len(diffs)} pairs; first component explains '
      f'{S[0]**2 / (S**2).sum():.1%} of the variance between them')

10 pairs; first component explains 28.7% of the variance between them


## Step 2: which words does it separate?

Project every word onto the gender direction. This is the standard bias measure,
and the words at each end are the test set for everything that follows.

In [3]:
DEFINITIONAL = {w for pair in PAIRS for w in pair}
bias = E @ g
candidates = [i for i, w in enumerate(words) if w not in DEFINITIONAL]
by_bias = sorted(candidates, key=lambda i: bias[i])

female_end, male_end = by_bias[:500], by_bias[-500:]
print('most male-associated :', [words[i] for i in male_end[-14:]])
print('most female-associated:', [words[i] for i in female_end[:14]])

selected = np.array(male_end + female_end)
labels = np.array([1] * 500 + [0] * 500)      # 1 = male-associated

most male-associated : ['businessman', 'cardinals', 'made', 'interim', 'politician', 'league', 'mlb', 'nfl', 'chairman', 'influential', 'appointed', 'premier', 'leader', 'elected']
most female-associated: ['diva', 'sisters', 'hers', 'mom', 'aunt', 'sassy', 'jennifer', 'natasha', 'stephanie', 'kournikova', 'stepmother', 'samantha', 'kuznetsova', 'caroline']


## Step 3: the tests

Three ways of asking whether gender is still in the vectors.

1. **Projection** onto `g`. This is the metric Bolukbasi optimises.
2. **k-means** into two clusters, scored against the original bias labels.
3. **SVM**, cross-validated, predicting the bias label from the vector.

Gonen & Goldberg's argument is that (1) is not evidence for (2) and (3).

In [4]:
def run_tests(vectors, name):
    X = vectors[selected]
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    km = KMeans(2, n_init=10, random_state=0).fit_predict(X)
    kmeans_acc = max((km == labels).mean(), (km != labels).mean())
    svm_acc = cross_val_score(SVC(kernel='rbf'), X, labels, cv=5).mean()
    projection = np.abs(vectors[selected] @ g).mean()
    print(f'{name:<22} mean |projection onto g| = {projection:.4f}'
          f'   k-means = {kmeans_acc:.1%}   SVM = {svm_acc:.1%}')


run_tests(E, 'ORIGINAL')

ORIGINAL               mean |projection onto g| = 0.2192   k-means = 100.0%   SVM = 100.0%


## Step 4: debias, then test again

Hard-debiasing, neutralise step: subtract the `g` component from every word that
is not gender-definitional. After this the projection is zero by construction.

In [5]:
E_debiased = E.copy()
neutral = np.array([i for i, w in enumerate(words) if w not in DEFINITIONAL])
E_debiased[neutral] -= np.outer(E_debiased[neutral] @ g, g)
E_debiased /= np.linalg.norm(E_debiased, axis=1, keepdims=True)

run_tests(E, 'ORIGINAL')
run_tests(E_debiased, 'AFTER DEBIASING')

ORIGINAL               mean |projection onto g| = 0.2192   k-means = 100.0%   SVM = 100.0%


AFTER DEBIASING        mean |projection onto g| = 0.0000   k-means = 80.5%   SVM = 98.2%


## Read the numbers

The projection is **exactly zero**. On the metric the method was designed to
optimise, the bias is perfectly removed.

An SVM still recovers the gender association at close to its original accuracy.
k-means still separates the two groups far above chance without being told what
to look for. The words that were gender-associated before are still sitting next
to each other afterwards.

Nothing was removed. The vectors were rotated so that the bias no longer lies
along the one axis anybody was measuring. Gonen & Goldberg's phrase for the
remaining structure is that the bias is "still there, and easily recoverable".

Their reported k-means figure is about 92.5%. This notebook gets a lower number
because it uses GloVe rather than word2vec and implements only the neutralise
step, not the full equalise procedure. The direction of the result is what
matters and it is not sensitive to those choices.

**Why this belongs in a compliance discussion.** It is the cleanest available
demonstration that *a bias metric going to zero is not evidence that bias is
gone*, especially when the intervention was designed against that metric. Any
Article 10 process that accepts "we removed the proxy variables" without an
independent recovery test is making the same mistake with less visibility.